# Preprocesado de la serie de precios de bovino

**Configuración del entorno.** Se importan las librerías y funciones auxiliares. Se activa *autoreload*, de manera que cambios en `utils` se reflejan sin reiniciar el *kernel*.

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import numpy as np
import pandas as pd

In [3]:
import sys
sys.path.append('..')  # carpeta src/

from utils.data_capture import connect, get_prices
from utils.preprocessing import uniform_ffill

**Captura de los datos.** Se extraen en formato tabular los precios semanales de vacuno de la colección `BinefarBovinePrice` de MongoDB. Cada fila corresponde a una fecha y cada columna a una categoría definida por el sexo, peso y clasificación (p. ej., `M_281/320KG_E`).

In [4]:
# Obtener el conjunto de datos en bruto
df_raw = pd.DataFrame()
with connect() as client:
    df_raw = get_prices(
        client, 'lonjaporcinodev.BinefarBovinePrice', pivot=True
    )
    df_raw['date'] = pd.to_datetime(df_raw['date'])
    df_raw = df_raw.set_index('date')

# Tamaño del conjunto de datos
n_rows, n_cols = df_raw.shape
print(f'Dimensiones:\n-núm. registros: {n_rows}\n-núm. variables: {n_cols}')

Dimensiones:
-núm. registros: 597
-núm. variables: 32


**Previsualización de los datos**. Se muestran los primeros registros para poder ver la estructura del conjunto de datos.

In [5]:
df_raw.head(n=5)

,F_180/220KG_E,F_180/220KG_O,F_180/220KG_R,F_180/220KG_U,F_221/260KG_E,F_221/260KG_O,F_221/260KG_R,F_221/260KG_U,F_261/300KG_E,F_261/300KG_O,...,M_321/370KG_R,M_321/370KG_U,M_<280KG_E,M_<280KG_O,M_<280KG_R,M_<280KG_U,M_>371KG_E,M_>371KG_O,M_>371KG_R,M_>371KG_U
date,,,,,,,,,,,,,,,,,,,,,
2014-12-29,4.56,4.0,4.29,4.46,4.52,3.91,4.24,4.4,4.41,3.77,...,3.55,3.82,4.05,3.36,3.63,3.91,3.87,3.11,3.46,3.72
2015-01-05,4.56,4.0,4.29,4.46,4.52,3.91,4.24,4.4,4.41,3.77,...,3.58,3.88,4.11,3.39,3.66,3.97,3.93,3.14,3.49,3.78
2015-01-12,4.56,4.0,4.29,4.46,4.52,3.91,4.24,4.4,4.41,3.77,...,3.61,3.91,4.14,3.42,3.69,4.00,3.96,3.17,3.52,3.81
2015-01-19,4.56,4.0,4.29,4.46,4.52,3.91,4.24,4.4,4.41,3.77,...,3.67,3.97,4.20,3.48,3.75,4.06,4.02,3.23,3.58,3.87
2015-01-26,4.56,4.0,4.29,4.46,4.52,3.91,4.24,4.4,4.41,3.77,...,3.67,3.97,4.20,3.48,3.75,4.06,4.02,3.23,3.58,3.87


**Orden cronológico y valores perdidos.** Se ordenan las filas por fecha. Se eliminan registros con todos los valores nulos y los nulos sueltos se rellenan con el último valor conocido de cada variable.

In [6]:
df_raw = df_raw.sort_index()
df_raw = df_raw.dropna(how='all').ffill()

**Intervalos cronológicos uniformes.** En los modelos que procesan secuencias temporales u ordenadas, la distancia temporal es clave y debe mantenerse constante (p. ej., no es recomendable tener registros separados por una semana y, de repente, incluir uno con un salto de cuatro semanas).

In [7]:
from collections import defaultdict

# Intervalos temporales
t_intervals = (df_raw.index.to_series().diff().dt.days).to_list()[1:]

# ¿Cuántas veces se produce cada intervalo?
t_counts = defaultdict(int)
for i in t_intervals:
    t_counts[i] += 1

for dias, count in t_counts.items():
    weeks, days = int(dias // 7), int(dias % 7)
    if days == 0: print(f'- {weeks} semanas: {count} filas')
    else: print(f'- {weeks} semanas y {days} días: {count} filas')

- 1 semanas: 578 filas
- 2 semanas: 16 filas
- 5 semanas: 1 filas
- 3 semanas: 1 filas


La mayoría de los registros están separados por una semana, aunque existen huecos de dos, tres y cinco semanas. Como solución habitual se añaden nuevos registros (semanas intermedias) usando ***forward fill***: rellenar los huecos copiando el último valor conocido hacia delante. Este método asume que el precio se mantiene estable entre registros de precios consecutivos y es adecuado para huecos de corta duración.

In [8]:
# Forward fill
df_raw = uniform_ffill(df_raw, freq='7D')

# Nuevas dimensiones
n_rows, n_cols = df_raw.shape
print(f'Dimensiones:\n- núm. registros: {n_rows}\n- núm. variables: {n_cols}\n')

# Comprobaciones
t_intervals = (df_raw.index.to_series().diff().dt.days).iloc[1:]
print('Comprobaciones:')
print(f'- ¿Todos los intervalos son de una semana?: {(t_intervals == 7).all()}')

Dimensiones:
- núm. registros: 619
- núm. variables: 32

Comprobaciones:
- ¿Todos los intervalos son de una semana?: True


**Exportación**. Se guarda el conjunto de datos preprocesado como `BinefarBovinePriceCleaned.csv`, que servirá de entrada para etapas posteriores.

In [9]:
PATH_CLEAN_DATA = '../../data/BinefarBovinePriceCleaned.csv'

df_raw.to_csv(PATH_CLEAN_DATA, sep=';')